## Libraries

In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, Model 

## GPU Configuration

In [2]:
# Check GPU availability
print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("Built with CUDA:", tf.test.is_built_with_cuda())

# Configure GPU memory growth (prevents TensorFlow from allocating all GPU memory at once)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"\n✓ GPU memory growth enabled for {len(gpus)} GPU(s)")
        
        # Optional: Enable mixed precision for better performance
        from tensorflow.keras import mixed_precision
        policy = mixed_precision.Policy('mixed_float16')
        mixed_precision.set_global_policy(policy)
        print(f"✓ Mixed precision enabled: {policy.name}")
        
    except RuntimeError as e:
        print(f"GPU configuration error: {e}")
else:
    print("\n⚠ No GPU detected - training will use CPU")

TensorFlow version: 2.21.0
GPU Available: []
Built with CUDA: False

⚠ No GPU detected - training will use CPU


## Data Cleaning

In [3]:
df = pd.read_parquet('../data/telemetry/silver/cda/Telemetry_Wide_With_States')
df.sort_values(['Unit', 'Fecha'], inplace=True)
df.drop_duplicates(subset=['Unit', 'Fecha'], keep='first', inplace=True)
df.drop(columns = ['Payload', 'EngOilFltr', 'AirFltr'], inplace=True)
print(f'Total rows: {len(df)/1000000:.3f}M')

Total rows: 4.184M


In [4]:
df.isna().sum()/len(df)*100

Fecha             0.000000
Unit              0.000000
Estado            0.000000
EstadoMaquina     0.000000
EstadoCarga       0.328164
GPSLat            1.451055
GPSLon            1.451055
GPSElevation      1.260667
CnkcasePres       1.663217
DiffLubePres     27.422675
DiffTemp          2.381144
EngCoolTemp       1.404850
EngOilPres        1.759307
EngSpd            0.000000
GroundSpd         0.084664
LtExhTemp         1.344782
LtFBrkTemp        1.482009
LtRBrkTemp        1.790883
RAftrclrTemp     17.150248
RtExhTemp         1.344901
RtFBrkTemp        7.242652
RtLtExhTemp      19.819972
RtRBrkTemp        1.481077
StrgOilTemp       2.440495
TCOutTemp         1.657672
TrnLubeTemp       3.150724
dtype: float64

In [5]:
margins = {
        # General
        'GPSLat' : (-30.4, -30.1),
        'GPSLon' : (-71.3, -70.9),
        'GPSElevation' : (400, 2000),
        'GroundSpd' : (0, 80),
        'EngSpd' : (0, 2500),
        # Engine
        "EngCoolTemp" : (30, 120),
        "RAftrclrTemp" : (10, 100),
        "EngOilPres" : (150, 700),
        "EngOilFltr" : (1, 50),
        "CnkcasePres" : (-1.5, 1.5),
        "RtLtExhTemp" : (-10, 10),
        "RtExhTemp" : (150, 750),
        "LtExhTemp" : (150, 750),
        # Transmission
        "DiffLubePres" : (0, 800),
        "DiffTemp" : (0, 150),
        "TrnLubeTemp" : (-5, 120),
        "TCOutTemp" : (30, 180),
        # Brakes
        "RtRBrkTemp" : (20, 200),
        "RtFBrkTemp" : (20, 200),
        "LtRBrkTemp" : (20, 200),
        "LtFBrkTemp" : (20, 200),
        # Direction
        'StrgOilTemp' : (-10, 150),
        
}

def clean_data(df_in, margins):
    """Function that uses the margin dict to clean the values -> all the values out of range are replaced by nan"""
    df = df_in.copy()
    for col, (lower, upper) in margins.items():
        if col in df.columns:
            df[col] = df[col].where((df[col] >= lower) & (df[col] <= upper), other=pd.NA)
    return df

df_cleaned = clean_data(df, margins)
print(f'Total rows after cleaning: {len(df_cleaned)/1000000:.3f}M')

num_cols = [col for col in margins.keys() if col in df_cleaned.columns]
df_cleaned.dropna(subset=num_cols, thresh=int(len(num_cols)/2), inplace=True)
df_cleaned.fillna({'EstadoMaquina':'ND', 'EstadoCarga':'Sin Carga'}, inplace=True)
df_cleaned.reset_index(drop=True, inplace=True)

print(f'Total rows after dropping rows with significant signals missing: {len(df_cleaned)/1000000:.3f}M')

df_cleaned.head()

Total rows after cleaning: 4.184M
Total rows after dropping rows with significant signals missing: 4.129M


,Fecha,Unit,Estado,EstadoMaquina,EstadoCarga,GPSLat,GPSLon,GPSElevation,CnkcasePres,DiffLubePres,...,LtFBrkTemp,LtRBrkTemp,RAftrclrTemp,RtExhTemp,RtFBrkTemp,RtLtExhTemp,RtRBrkTemp,StrgOilTemp,TCOutTemp,TrnLubeTemp
0,2025-01-01 00:02:00,T_10,ND,ND,Sin Carga,-30.254324,-71.091386,998.488372,0.151770,NaN,...,79.666667,78.000000,35.366667,250.982564,80.000000,-6.723517,78.833333,68.000000,80.739394,86.555556
1,2025-01-01 00:03:00,T_10,ND,ND,Sin Carga,-30.254331,-71.091389,996.500000,0.138828,NaN,...,79.500000,78.000000,35.275000,234.736923,79.750000,-7.042637,78.875000,68.000000,80.554545,86.416667
2,2025-01-01 00:10:00,T_10,ND,ND,Sin Carga,-30.254054,-71.091847,1000.935136,NaN,NaN,...,77.500000,76.422222,NaN,225.460740,76.833333,NaN,76.666667,63.166667,NaN,NaN
3,2025-01-01 00:11:00,T_10,ND,ND,Sin Carga,-30.254104,-71.093598,982.186666,-0.083427,NaN,...,76.800000,77.495238,35.377778,213.910555,76.416667,-5.962302,77.250000,63.375000,80.651961,83.444444
4,2025-01-01 00:12:00,T_10,ND,ND,Sin Carga,-30.252903,-71.096539,939.699995,-0.025851,51.308511,...,76.629545,80.878571,37.033333,210.314722,77.229167,-1.987033,79.808824,64.125000,80.670789,83.583333


In [6]:
df_cleaned.isna().sum()/len(df_cleaned)*100

Fecha             0.000000
Unit              0.000000
Estado            0.000000
EstadoMaquina     0.000000
EstadoCarga       0.000000
GPSLat            1.569995
GPSLon            1.569971
GPSElevation      1.263107
CnkcasePres       0.399207
DiffLubePres     26.549289
DiffTemp          1.128843
EngCoolTemp       0.262134
EngOilPres        0.774657
EngSpd            0.000145
GroundSpd         0.092851
LtExhTemp         3.452280
LtFBrkTemp        0.299720
LtRBrkTemp        0.600506
RAftrclrTemp     16.114443
RtExhTemp         4.078360
RtFBrkTemp        7.504727
RtLtExhTemp      31.539718
RtRBrkTemp        0.274388
StrgOilTemp       1.191155
TCOutTemp         1.326703
TrnLubeTemp       1.898922
dtype: float64

In [7]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# =========================
# Config
# =========================
UNIT_COL = "Unit"
TIME_COL = "Fecha"
CAT_COLS = ["EstadoMaquina", "EstadoCarga"]   # se mantienen
FREQ = "1min"
INTERP_LIMIT = 10

df_final = df_cleaned.copy()

# Asegurar orden
df_final = df_final.sort_values([UNIT_COL, TIME_COL]).reset_index(drop=True)

# Detectar columnas numéricas reales a interpolar
# Convertir accidental object/string numeric columns
for c in num_cols:
    if df_final[c].dtype == "object" or pd.api.types.is_string_dtype(df_final[c]):
        df_final[c] = pd.to_numeric(df_final[c], errors="coerce")


def plot_ts(df_original: pd.DataFrame, df_end: pd.DataFrame, num_cols: list, time_col: str = TIME_COL):
    """
    Plotea una señal por figura usando seaborn/matplotlib.
    Arriba: versión original.
    Abajo: versión reindexada/interpolada.
    """
    import matplotlib.pyplot as plt
    import seaborn as sns

    for signal in num_cols:
        fig, axes = plt.subplots(
            nrows=2,
            ncols=1,
            figsize=(14, 8),
            sharex=True
        )

        sns.scatterplot(
            data=df_original,
            x=time_col,
            y=signal,
            ax=axes[0],
            legend=False
        )
        axes[0].set_title(f"Original {signal}")
        axes[0].set_ylabel(signal)
        axes[0].grid(True, alpha=0.3)

        sns.scatterplot(
            data=df_end,
            x=time_col,
            y=signal,
            ax=axes[1],
            legend=False
        )
        axes[1].set_title(f"Interpolated {signal}")
        axes[1].set_xlabel(time_col)
        axes[1].set_ylabel(signal)
        axes[1].grid(True, alpha=0.3)

        fig.suptitle(f"Señal: {signal}", fontsize=14)
        plt.tight_layout()
        plt.show()


def reindex_and_interpolate_unit(
    unit_df: pd.DataFrame,
    num_cols: list,
    freq: str = FREQ,
    interp_limit: int = INTERP_LIMIT,
    viz: bool = False
) -> pd.DataFrame:
    """
    Reindexa la serie completa de una unidad a 1 minuto e interpola solo huecos internos
    de hasta `interp_limit` puntos consecutivos usando tiempo.

    Notas:
    - No usa cycle_id
    - Mantiene continuidad temporal completa por unidad
    - No rellena extremos por limit_area='inside'
    - No cruzará gaps muy largos porque limit impide imputar más de N puntos seguidos
    """
    out = unit_df.copy().sort_values(TIME_COL).reset_index(drop=True)

    if out.empty:
        return out.copy()

    original_timestamps = set(out[TIME_COL])

    full_time_index = pd.date_range(
        start=out[TIME_COL].min(),
        end=out[TIME_COL].max(),
        freq=freq
    )

    out = out.set_index(TIME_COL).reindex(full_time_index)
    out.index.name = TIME_COL

    # Reconstruir metadata fija
    out[UNIT_COL] = unit_df[UNIT_COL].iloc[0]

    # Reconstruir categóricas
    for c in CAT_COLS:
        if c in unit_df.columns:
            out[c] = out[c].ffill().bfill()

    valid_cols = [c for c in num_cols if c in out.columns]

    # Guardar posiciones faltantes antes de interpolar
    before_na = out[valid_cols].isna()

    # Flag de filas creadas por reindex
    out["created_by_reindex"] = (~out.index.isin(original_timestamps)).astype("int8")

    # Interpolar numéricas
    out[valid_cols] = out[valid_cols].interpolate(
        method="time",
        limit=interp_limit,
        limit_area="inside"
    )

    # Flag: al menos una señal fue imputada en esa fila
    out["imputed_any"] = (
        (before_na & ~out[valid_cols].isna()).any(axis=1)
    ).astype("int8")

    # Conteo de señales imputadas por fila
    out["n_imputed_signals"] = (
        before_na & ~out[valid_cols].isna()
    ).sum(axis=1).astype("int16")

    # Gap bruto respecto al timestamp anterior original/reindexado
    out["time_gap_min"] = (
        pd.Series(out.index, index=out.index).diff().dt.total_seconds().div(60)
    )

    if viz:
        n_values_interpolated_per_signal_dict = (
            (before_na & ~out[valid_cols].isna()).sum().to_dict()
        )
        max_imputed_values = max(n_values_interpolated_per_signal_dict.values()) if n_values_interpolated_per_signal_dict else 0

        if len(unit_df) > 0 and max_imputed_values / len(unit_df) > 0.2:
            random_bool = np.random.choice([False, True], p=[0.996, 0.004])
            if random_bool:
                interpolated_cols = [
                    col for col, n_imputed in n_values_interpolated_per_signal_dict.items()
                    if n_imputed / len(unit_df) > 0.05
                ]
                relevant_n_values = {
                    col: n for col, n in n_values_interpolated_per_signal_dict.items()
                    if col in interpolated_cols
                }
                print(f"Unit {unit_df[UNIT_COL].iloc[0]}: Imputed values per signal: {relevant_n_values}")
                plot_ts(
                    df_original=unit_df,
                    df_end=out.reset_index(),
                    num_cols=interpolated_cols
                )

    return out.reset_index()


# =========================
# Aplicar por unidad
# =========================
unit_results = []

for unit_id, unit_df in tqdm(df_final.groupby(UNIT_COL, sort=False), desc="Processing units"):
    unit_out = reindex_and_interpolate_unit(
        unit_df=unit_df,
        num_cols=num_cols,
        freq=FREQ,
        interp_limit=INTERP_LIMIT,
        viz=False
    )
    unit_results.append(unit_out)

cleaned_df = pd.concat(unit_results, ignore_index=True) if unit_results else df_final.head(0).copy()

Processing units:   0%|          | 0/11 [00:00<?, ?it/s]

In [8]:
print(f'Total rows after imputing: {len(cleaned_df)/1000000:.3f}M')

cleaned_df.isna().sum()/len(cleaned_df)*100

Total rows after imputing: 6.182M


Fecha                  0.000000
Unit                   0.000000
Estado                33.202559
EstadoMaquina          0.000000
EstadoCarga            0.000000
GPSLat                30.233950
GPSLon                30.233950
GPSElevation          30.215541
CnkcasePres           29.544216
DiffLubePres          35.033478
DiffTemp              29.949382
EngCoolTemp           29.561945
EngOilPres            29.939887
EngSpd                29.386604
GroundSpd             29.400370
LtExhTemp             31.059183
LtFBrkTemp            29.573463
LtRBrkTemp            29.739503
RAftrclrTemp          40.762580
RtExhTemp             30.987018
RtFBrkTemp            34.639878
RtLtExhTemp           38.809218
RtRBrkTemp            29.567365
StrgOilTemp           30.181586
TCOutTemp             30.316210
TrnLubeTemp           30.614868
created_by_reindex     0.000000
imputed_any            0.000000
n_imputed_signals      0.000000
time_gap_min           0.000178
dtype: float64

## Model

In [9]:
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import List, Optional, Dict, Any, Tuple

from sklearn.preprocessing import RobustScaler, OneHotEncoder
from numpy.lib.stride_tricks import sliding_window_view

def _rolling_mean_1d(arr: np.ndarray, win: int) -> np.ndarray:
    """
    Rolling mean for 1D numeric array.
    Returns length = len(arr) - win + 1
    """
    arr = np.asarray(arr, dtype=np.float64)
    c = np.empty(len(arr) + 1, dtype=np.float64)
    c[0] = 0.0
    np.cumsum(arr, out=c[1:])
    return (c[win:] - c[:-win]) / win


def _ensure_window_axis_order(x: np.ndarray, win: int) -> np.ndarray:
    """
    sliding_window_view on axis=0 may return either:
    (n_windows, win, n_features)
    or
    (n_windows, n_features, win)
    depending on NumPy behavior/version.
    Normalize to (n_windows, win, n_features).
    """
    if x.ndim != 3:
        raise ValueError(f"Expected 3D array, got shape {x.shape}")

    if x.shape[1] == win:
        return x
    if x.shape[2] == win:
        return np.swapaxes(x, 1, 2)

    raise ValueError(f"Could not identify window axis in shape {x.shape}")


def _fill_by_unit_fast(arr_2d: np.ndarray, fill_value: float) -> np.ndarray:
    """
    Fast compromise:
    Fill missing values per unit/segment before windowing.
    This is much faster than filling each window separately.

    Behavior note:
    This is not exactly identical to per-window ffill/bfill because values
    can propagate across a longer unit segment, not only within each window.
    In most telemetry use cases this is acceptable and much faster.
    """
    df = pd.DataFrame(arr_2d)
    return df.ffill().bfill().fillna(fill_value).to_numpy()

@dataclass
class WindowingConfig:
    unit_col: str = UNIT_COL
    time_col: str = TIME_COL

    numeric_cols: Optional[List[str]] = None
    categorical_cols: Optional[List[str]] = None

    # train windowing
    train_window_size: int = 60
    train_step_size: int = 1

    # predict windowing
    predict_window_size: int = 60   # 1 hour if frequency = 1 minute

    # train filters
    min_numeric_coverage: float = 0.80
    min_row_coverage: float = 0.80
    max_created_fraction: Optional[float] = None
    max_imputed_fraction: Optional[float] = None

    # fill values after scaling
    train_fill_value: float = 0.0
    predict_fill_value: float = -10.0

    output_dtype: str = "float32"


class LSTMAutoencoderPreprocessor:
    """
    Preprocessor for:
    - X = [numeric_scaled + categorical_encoded]
    - y = numeric_scaled only

    Supports:
    - fit() on train data
    - transform_rows()
    - make_train_windows() -> sliding windows
    - make_predict_windows() -> 1-hour windows per unit
    """

    def __init__(self, config: WindowingConfig):
        self.config = config

        self.numeric_scaler = None
        self.ohe = None

        self.input_feature_names_: Optional[List[str]] = None
        self.target_feature_names_: Optional[List[str]] = None

        self.numeric_fill_values_: Optional[Dict[str, float]] = None
        self.is_fitted_: bool = False

    # =====================================================
    # Public API
    # =====================================================
    def fit(self, df: pd.DataFrame) -> "LSTMAutoencoderPreprocessor":
        df = self._validate_and_prepare_input(df)

        numeric_cols = self.config.numeric_cols or []
        categorical_cols = self.config.categorical_cols or []

        self.numeric_fill_values_ = {
            col: df[col].median(skipna=True) if col in df.columns else 0.0
            for col in numeric_cols
        }

        # Fit numeric scaler
        self.numeric_scaler = RobustScaler()
        if numeric_cols:
            num_fit = df[numeric_cols].copy()
            for col in numeric_cols:
                num_fit[col] = num_fit[col].fillna(self.numeric_fill_values_[col])
            self.numeric_scaler.fit(num_fit)

        # Fit OHE
        self.ohe = OneHotEncoder(
            handle_unknown="ignore",
            drop="first",
            sparse_output=False
        )
        if categorical_cols:
            cat_fit = df[categorical_cols].copy().astype("string").fillna("__missing__")
            self.ohe.fit(cat_fit)

        self.target_feature_names_ = list(numeric_cols)
        self.input_feature_names_ = self._build_input_feature_names()

        self.is_fitted_ = True
        return self

    def transform_rows(self, df: pd.DataFrame) -> pd.DataFrame:
        self._check_is_fitted()
        df = self._validate_and_prepare_input(df)

        base_cols = [self.config.unit_col, self.config.time_col]
        extra_cols = [
            c for c in ["created_by_reindex", "imputed_any", "n_imputed_signals"]
            if c in df.columns
        ]

        num_df = self._transform_numeric(df)
        cat_df = self._transform_categorical(df)

        out_parts = [df[base_cols + extra_cols].reset_index(drop=True)]
        if not num_df.empty:
            out_parts.append(num_df.reset_index(drop=True))
        if not cat_df.empty:
            out_parts.append(cat_df.reset_index(drop=True))

        out = pd.concat(out_parts, axis=1, copy=False)
        return out

    def fit_transform_rows(self, df: pd.DataFrame) -> pd.DataFrame:
        self.fit(df)
        return self.transform_rows(df)

    def make_train_windows(
        self,
        raw_df: pd.DataFrame,
        transformed_df: pd.DataFrame,
        return_metadata: bool = True
    ):
        self._check_is_fitted()

        raw_df = self._validate_and_prepare_input(raw_df)
        transformed_df = transformed_df.copy()

        unit_col = self.config.unit_col
        time_col = self.config.time_col
        win = self.config.train_window_size
        step = self.config.train_step_size

        input_cols = self.input_feature_names_
        target_cols = self.target_feature_names_
        numeric_cols = self.config.numeric_cols or []

        # Ensure same ordering
        raw_df = raw_df.sort_values([unit_col, time_col]).reset_index(drop=True)
        transformed_df = transformed_df.sort_values([unit_col, time_col]).reset_index(drop=True)

        if len(raw_df) != len(transformed_df):
            raise ValueError("raw_df and transformed_df have different number of rows.")

        if not raw_df[[unit_col, time_col]].equals(transformed_df[[unit_col, time_col]]):
            raise ValueError("raw_df and transformed_df row alignment mismatch.")

        unit_arr = raw_df[unit_col].to_numpy()
        time_arr = raw_df[time_col].to_numpy()

        raw_num = raw_df[numeric_cols].to_numpy(dtype=np.float32, copy=False) if numeric_cols else np.empty((len(raw_df), 0), dtype=np.float32)
        X_all = transformed_df[input_cols].to_numpy(dtype=np.float32, copy=True)
        y_all = transformed_df[target_cols].to_numpy(dtype=np.float32, copy=True)

        created_arr = raw_df["created_by_reindex"].to_numpy(dtype=np.float32, copy=False) if "created_by_reindex" in raw_df.columns else None
        imputed_arr = raw_df["imputed_any"].to_numpy(dtype=np.float32, copy=False) if "imputed_any" in raw_df.columns else None

        # Find contiguous unit segments
        change = np.r_[True, unit_arr[1:] != unit_arr[:-1]]
        starts = np.flatnonzero(change)
        ends = np.r_[starts[1:], len(unit_arr)]

        X_seq = []
        y_seq = []
        meta_rows = []

        n_signals = len(numeric_cols)

        for s, e in zip(starts, ends):
            unit = unit_arr[s]
            n = e - s
            if n < win:
                continue

            raw_num_u = raw_num[s:e]
            X_u = X_all[s:e]
            y_u = y_all[s:e]
            time_u = time_arr[s:e]

            # Faster fill once per unit instead of once per window
            X_u_filled = _fill_by_unit_fast(X_u, fill_value=self.config.train_fill_value)
            y_u_filled = _fill_by_unit_fast(y_u, fill_value=self.config.train_fill_value)

            # Window views
            X_view = sliding_window_view(X_u_filled, window_shape=win, axis=0)
            y_view = sliding_window_view(y_u_filled, window_shape=win, axis=0)
            X_view = _ensure_window_axis_order(X_view, win)
            y_view = _ensure_window_axis_order(y_view, win)

            # Rolling quality
            if n_signals > 0:
                observed = ~np.isnan(raw_num_u)
                observed_count_per_row = observed.sum(axis=1).astype(np.float32)
                row_has_any = observed.any(axis=1).astype(np.float32)

                numeric_coverage = _rolling_mean_1d(observed_count_per_row, win) / n_signals
                row_coverage = _rolling_mean_1d(row_has_any, win)
            else:
                numeric_coverage = np.ones(n - win + 1, dtype=np.float32)
                row_coverage = np.ones(n - win + 1, dtype=np.float32)

            valid = (
                (numeric_coverage >= self.config.min_numeric_coverage) &
                (row_coverage >= self.config.min_row_coverage)
            )

            created_fraction = None
            if created_arr is not None:
                created_fraction = _rolling_mean_1d(created_arr[s:e], win)
                if self.config.max_created_fraction is not None:
                    valid &= (created_fraction <= self.config.max_created_fraction)

            imputed_fraction = None
            if imputed_arr is not None:
                imputed_fraction = _rolling_mean_1d(imputed_arr[s:e], win)
                if self.config.max_imputed_fraction is not None:
                    valid &= (imputed_fraction <= self.config.max_imputed_fraction)

            # Apply step
            idx = np.arange(0, n - win + 1, step)
            valid_idx = idx[valid[idx]]

            if len(valid_idx) == 0:
                continue

            X_sel = X_view[valid_idx]
            y_sel = y_view[valid_idx]

            # final safety
            finite_mask = np.isfinite(X_sel).all(axis=(1, 2)) & np.isfinite(y_sel).all(axis=(1, 2))
            if not finite_mask.all():
                X_sel = X_sel[finite_mask]
                y_sel = y_sel[finite_mask]
                valid_idx = valid_idx[finite_mask]

            if len(valid_idx) == 0:
                continue

            X_seq.append(X_sel.astype(self.config.output_dtype, copy=False))
            y_seq.append(y_sel.astype(self.config.output_dtype, copy=False))

            if return_metadata:
                for start_idx in valid_idx:
                    row = {
                        "Unit": unit,
                        "window_type": "train_sliding",
                        "start_idx": int(start_idx),
                        "end_idx_exclusive": int(start_idx + win),
                        "start_time": time_u[start_idx],
                        "end_time": time_u[start_idx + win - 1],
                        "numeric_coverage": float(numeric_coverage[start_idx]),
                        "row_coverage": float(row_coverage[start_idx]),
                    }
                    if created_fraction is not None:
                        row["created_fraction"] = float(created_fraction[start_idx])
                    if imputed_fraction is not None:
                        row["imputed_fraction"] = float(imputed_fraction[start_idx])
                    meta_rows.append(row)

        X = np.concatenate(X_seq, axis=0).astype(self.config.output_dtype) if X_seq else np.empty(
            (0, win, len(input_cols)), dtype=self.config.output_dtype
        )

        y = np.concatenate(y_seq, axis=0).astype(self.config.output_dtype) if y_seq else np.empty(
            (0, win, len(target_cols)), dtype=self.config.output_dtype
        )

        meta = pd.DataFrame(meta_rows) if return_metadata else None
        return X, y, meta
    
    def make_predict_windows(
        self,
        raw_df: pd.DataFrame,
        transformed_df: pd.DataFrame,
        return_metadata: bool = True
    ):
        self._check_is_fitted()

        raw_df = self._validate_and_prepare_input(raw_df)
        transformed_df = transformed_df.copy()

        unit_col = self.config.unit_col
        time_col = self.config.time_col
        win = self.config.predict_window_size

        input_cols = self.input_feature_names_
        target_cols = self.target_feature_names_

        raw_df = raw_df.sort_values([unit_col, time_col]).reset_index(drop=True)
        transformed_df = transformed_df.sort_values([unit_col, time_col]).reset_index(drop=True)

        if len(raw_df) != len(transformed_df):
            raise ValueError("raw_df and transformed_df have different number of rows.")

        if not raw_df[[unit_col, time_col]].equals(transformed_df[[unit_col, time_col]]):
            raise ValueError("raw_df and transformed_df row alignment mismatch.")

        unit_arr = raw_df[unit_col].to_numpy()
        time_arr = raw_df[time_col].to_numpy()

        X_all = transformed_df[input_cols].to_numpy(dtype=np.float32, copy=True)
        y_all = transformed_df[target_cols].to_numpy(dtype=np.float32, copy=True)

        created_arr = raw_df["created_by_reindex"].to_numpy(dtype=np.float32, copy=False) if "created_by_reindex" in raw_df.columns else None
        imputed_arr = raw_df["imputed_any"].to_numpy(dtype=np.float32, copy=False) if "imputed_any" in raw_df.columns else None

        change = np.r_[True, unit_arr[1:] != unit_arr[:-1]]
        starts = np.flatnonzero(change)
        ends = np.r_[starts[1:], len(unit_arr)]

        X_seq = []
        y_seq = []
        meta_rows = []

        for s, e in zip(starts, ends):
            unit = unit_arr[s]
            n = e - s
            if n < win:
                continue

            n_complete_windows = n // win
            usable = n_complete_windows * win
            if usable == 0:
                continue

            X_u = X_all[s:s + usable]
            y_u = y_all[s:s + usable]
            time_u = time_arr[s:s + usable]

            X_u_filled = _fill_by_unit_fast(X_u, fill_value=self.config.predict_fill_value)
            y_u_filled = _fill_by_unit_fast(y_u, fill_value=self.config.predict_fill_value)

            X_blocks = X_u_filled.reshape(n_complete_windows, win, -1)
            y_blocks = y_u_filled.reshape(n_complete_windows, win, -1)

            finite_mask = np.isfinite(X_blocks).all(axis=(1, 2)) & np.isfinite(y_blocks).all(axis=(1, 2))
            if not finite_mask.all():
                X_blocks = X_blocks[finite_mask]
                y_blocks = y_blocks[finite_mask]

            if len(X_blocks) == 0:
                continue

            X_seq.append(X_blocks.astype(self.config.output_dtype, copy=False))
            y_seq.append(y_blocks.astype(self.config.output_dtype, copy=False))

            if return_metadata:
                for hour_idx in range(n_complete_windows):
                    start_idx = hour_idx * win
                    end_idx = start_idx + win
                    row = {
                        "Unit": unit,
                        "window_type": "predict_hour_block",
                        "hour_idx": hour_idx,
                        "start_idx": start_idx,
                        "end_idx_exclusive": end_idx,
                        "start_time": time_u[start_idx],
                        "end_time": time_u[end_idx - 1],
                        "n_rows": win,
                    }
                    if created_arr is not None:
                        row["created_fraction"] = float(created_arr[s + start_idx:s + end_idx].mean())
                    else:
                        row["created_fraction"] = np.nan

                    if imputed_arr is not None:
                        row["imputed_fraction"] = float(imputed_arr[s + start_idx:s + end_idx].mean())
                    else:
                        row["imputed_fraction"] = np.nan

                    meta_rows.append(row)

        X = np.concatenate(X_seq, axis=0).astype(self.config.output_dtype) if X_seq else np.empty(
            (0, win, len(input_cols)), dtype=self.config.output_dtype
        )
        y = np.concatenate(y_seq, axis=0).astype(self.config.output_dtype) if y_seq else np.empty(
            (0, win, len(target_cols)), dtype=self.config.output_dtype
        )

        meta = pd.DataFrame(meta_rows) if return_metadata else None
        return X, y, meta
    
    def fit_transform_train(
        self,
        df_train: pd.DataFrame,
        return_metadata: bool = True
    ) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame, Optional[pd.DataFrame]]:
        """
        Fit on train -> transform rows -> create train windows
        """
        tr_rows = self.fit_transform_rows(df_train)
        X, y, meta = self.make_train_windows(
            raw_df=df_train,
            transformed_df=tr_rows,
            return_metadata=return_metadata
        )
        return X, y, tr_rows, meta

    def transform_predict(
        self,
        df_test: pd.DataFrame,
        return_metadata: bool = True
    ) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame, Optional[pd.DataFrame]]:
        """
        Transform test -> create hourly predict windows
        """
        tr_rows = self.transform_rows(df_test)
        X, y, meta = self.make_predict_windows(
            raw_df=df_test,
            transformed_df=tr_rows,
            return_metadata=return_metadata
        )
        
        print(f'Shape of predict windows: {X.shape}, {y.shape}')
        
        return X, y, tr_rows, meta

    # =====================================================
    # Internal methods
    # =====================================================
    def _validate_and_prepare_input(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        required_cols = [self.config.unit_col, self.config.time_col]
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")

        if self.config.numeric_cols is None:
            raise ValueError("config.numeric_cols must be provided explicitly.")

        if self.config.categorical_cols is None:
            self.config.categorical_cols = []

        if not pd.api.types.is_datetime64_any_dtype(df[self.config.time_col]):
            df[self.config.time_col] = pd.to_datetime(df[self.config.time_col], errors="coerce")

        df = df.dropna(subset=[self.config.unit_col, self.config.time_col])
        df = df.sort_values([self.config.unit_col, self.config.time_col]).reset_index(drop=True)

        for col in self.config.numeric_cols:
            if col in df.columns and (
                df[col].dtype == "object" or pd.api.types.is_string_dtype(df[col])
            ):
                df[col] = pd.to_numeric(df[col], errors="coerce")

        for col in self.config.categorical_cols:
            if col in df.columns:
                df[col] = df[col].astype("string")

        return df

    def _transform_numeric(self, df: pd.DataFrame) -> pd.DataFrame:
        numeric_cols = self.config.numeric_cols or []
        if not numeric_cols:
            return pd.DataFrame(index=df.index)

        num_df = df[numeric_cols].copy()

        # Keep original NaN mask
        nan_mask = num_df.isna()

        # Temporary fill only to apply scaler
        fill_values = self.numeric_fill_values_ or {c: 0.0 for c in numeric_cols}
        num_temp = num_df.copy()
        for c in numeric_cols:
            num_temp[c] = num_temp[c].fillna(fill_values.get(c, 0.0))

        scaled = pd.DataFrame(
            self.numeric_scaler.transform(num_temp),
            columns=numeric_cols,
            index=df.index
        )

        # Restore original NaNs so train/predict can handle them differently later
        scaled[nan_mask] = np.nan

        return scaled

    def _transform_categorical(self, df: pd.DataFrame) -> pd.DataFrame:
        categorical_cols = self.config.categorical_cols or []
        if not categorical_cols:
            return pd.DataFrame(index=df.index)

        cat_df = df[categorical_cols].copy().astype("string").fillna("__missing__")
        arr = self.ohe.transform(cat_df)
        cols = self.ohe.get_feature_names_out(categorical_cols).tolist()
        return pd.DataFrame(arr, columns=cols, index=df.index)

    def _build_input_feature_names(self) -> List[str]:
        names = list(self.config.numeric_cols or [])
        if self.config.categorical_cols:
            names.extend(
                self.ohe.get_feature_names_out(self.config.categorical_cols).tolist()
            )
        return names

    def _raw_window_quality_ok(self, raw_window: pd.DataFrame) -> Tuple[bool, Dict[str, Any]]:
        """
        Quality filtering for training windows only, using raw numeric data.
        """
        signal_cols = self.config.numeric_cols or []
        observed = raw_window[signal_cols].notna()

        numeric_coverage = float(observed.mean().mean()) if len(signal_cols) > 0 else 1.0
        row_coverage = float(observed.any(axis=1).mean()) if len(signal_cols) > 0 else 1.0

        info = {
            "numeric_coverage": numeric_coverage,
            "row_coverage": row_coverage,
        }

        if numeric_coverage < self.config.min_numeric_coverage:
            return False, info

        if row_coverage < self.config.min_row_coverage:
            return False, info

        if "created_by_reindex" in raw_window.columns:
            created_fraction = float(raw_window["created_by_reindex"].mean())
            info["created_fraction"] = created_fraction
            if self.config.max_created_fraction is not None and created_fraction > self.config.max_created_fraction:
                return False, info

        if "imputed_any" in raw_window.columns:
            imputed_fraction = float(raw_window["imputed_any"].mean())
            info["imputed_fraction"] = imputed_fraction
            if self.config.max_imputed_fraction is not None and imputed_fraction > self.config.max_imputed_fraction:
                return False, info

        return True, info

    def _final_fill_window(self, df_window: pd.DataFrame, fill_value: float) -> pd.DataFrame:
        """
        Final guard before tensor conversion.
        Keeps current values and fills remaining NaNs with chosen constant.
        """
        out = df_window.copy()
        out = out.ffill().bfill().fillna(fill_value)
        return out

    def _check_is_fitted(self):
        if not self.is_fitted_:
            raise RuntimeError("Preprocessor is not fitted yet. Call fit() first.")
        

In [10]:
SIGNAL_COLS = [
          "EngCoolTemp",
          "RAftrclrTemp",
          "EngOilPres",
          "CnkcasePres",
          "RtExhTemp",
          "LtExhTemp"
]

config = WindowingConfig(
    unit_col=UNIT_COL,
    time_col=TIME_COL,
    numeric_cols=SIGNAL_COLS,
    categorical_cols=["EstadoMaquina", "EstadoCarga"],

    train_window_size=60,
    train_step_size=1,
    predict_window_size=60,

    min_numeric_coverage=0.85,
    min_row_coverage=0.95,
    max_created_fraction=0.08,
    max_imputed_fraction=0.22,

    train_fill_value=0.0,
    predict_fill_value=-10.0,
)



In [11]:
preprocessor = LSTMAutoencoderPreprocessor(config)

WEEKS_TO_TEST = 8
df_train = cleaned_df[cleaned_df[TIME_COL] < cleaned_df[TIME_COL].max() - pd.Timedelta(weeks=WEEKS_TO_TEST)]
df_test = cleaned_df[cleaned_df[TIME_COL] >= cleaned_df[TIME_COL].max() - pd.Timedelta(weeks=WEEKS_TO_TEST)]

In [ ]:
X_train, y_train, train_rows, train_meta = preprocessor.fit_transform_train(df_train)

print(f'Shape of X_train: {X_train.shape}')  # (n_windows, 60, n_input_features)
print(f'Shape of y_train: {y_train.shape}')  # (n_windows, 60, n_numeric_features)

# null recount
print(f'Nulls in X_train: {np.isnan(X_train).sum()}')
print(f'Nulls in y_train: {np.isnan(y_train).sum()}')

In [ ]:
X_pred, y_pred, pred_rows, pred_meta = preprocessor.transform_predict(df_test)

print(X_pred.shape)
print(y_pred.shape)
print(pred_meta.head())

In [13]:
from dataclasses import dataclass, asdict
from typing import Optional, Dict, Any

@dataclass
class LSTMAEModelConfig:
    latent_dim: int = 8
    encoder_lstm_1: int = 32
    encoder_lstm_2: int = 16
    decoder_lstm_1: int = 16
    decoder_lstm_2: int = 32

    dropout_rate: float = 0.2
    learning_rate: float = 1e-3

    batch_size: int = 32
    epochs: int = 50
    validation_split: float = 0.2
    early_stopping_patience: int = 5

    loss: str = "mse"
    metrics: tuple = ()
    

import os
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from typing import Tuple, Optional
from tensorflow.keras import Model, layers


class LSTMAutoencoderService:
    """
    End-to-end service for:
    - building the model
    - training
    - saving/loading artifacts
    - running prediction
    - computing reconstruction error
    """

    def __init__(
        self,
        preprocessor,
        model_config: LSTMAEModelConfig,
        model: Optional[tf.keras.Model] = None
    ):
        self.preprocessor = preprocessor
        self.model_config = model_config
        self.model = model
        self.history_ = None

    # =====================================================
    # Model building
    # =====================================================
    def build_model(self) -> tf.keras.Model:
        """
        Build seq2seq LSTM autoencoder:
        X: (window, input_features)
        y: (window, numeric_features)
        """
        if not self.preprocessor.is_fitted_:
            raise RuntimeError("Preprocessor must be fitted before building the model.")

        seq_len = self.preprocessor.config.train_window_size
        n_input_features = len(self.preprocessor.input_feature_names_)
        n_output_features = len(self.preprocessor.target_feature_names_)

        cfg = self.model_config

        encoder_inputs = layers.Input(
            shape=(seq_len, n_input_features),
            name="encoder_inputs"
        )

        x = layers.Masking(mask_value=self.preprocessor.config.predict_fill_value)(encoder_inputs)
        x = layers.LSTM(cfg.encoder_lstm_1, return_sequences=True, name="enc_lstm_1")(x)
        x = layers.Dropout(cfg.dropout_rate, name="enc_dropout_1")(x)
        x = layers.LSTM(cfg.encoder_lstm_2, return_sequences=False, name="enc_lstm_2")(x)
        x = layers.Dropout(cfg.dropout_rate, name="enc_dropout_2")(x)

        latent = layers.Dense(cfg.latent_dim, activation="linear", name="latent_vector")(x)

        x = layers.RepeatVector(seq_len, name="repeat_vector")(latent)
        x = layers.LSTM(cfg.decoder_lstm_1, return_sequences=True, name="dec_lstm_1")(x)
        x = layers.Dropout(cfg.dropout_rate, name="dec_dropout_1")(x)
        x = layers.LSTM(cfg.decoder_lstm_2, return_sequences=True, name="dec_lstm_2")(x)

        decoder_outputs = layers.TimeDistributed(
            layers.Dense(n_output_features),
            name="reconstructed_numeric"
        )(x)

        model = Model(encoder_inputs, decoder_outputs, name="lstm_autoencoder")

        optimizer = tf.keras.optimizers.Adam(learning_rate=cfg.learning_rate)
        model.compile(
            optimizer=optimizer,
            loss=cfg.loss,
            metrics=list(cfg.metrics)
        )

        self.model = model
        return model

    # =====================================================
    # Training
    # =====================================================
    def fit(
        self,
        df_train: pd.DataFrame,
        verbose: int = 1
    ) -> Dict[str, Any]:
        """
        Fit preprocessor + create train windows + train model.
        """
        X_train, y_train, transformed_rows, train_meta = self.preprocessor.fit_transform_train(
            df_train,
            return_metadata=True
        )

        if X_train.shape[0] == 0:
            raise ValueError("No valid training windows were generated.")

        if self.model is None:
            self.build_model()

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                patience=self.model_config.early_stopping_patience,
                restore_best_weights=True,
                monitor="val_loss"
            )
        ]

        history = self.model.fit(
            X_train,
            y_train,
            epochs=self.model_config.epochs,
            batch_size=self.model_config.batch_size,
            validation_split=self.model_config.validation_split,
            callbacks=callbacks,
            verbose=verbose,
            shuffle=True
        )

        self.history_ = history.history

        return {
            "X_train_shape": X_train.shape,
            "y_train_shape": y_train.shape,
            "n_train_windows": int(X_train.shape[0]),
            "history": self.history_,
            "train_meta": train_meta,
            "transformed_rows": transformed_rows,
        }

    # =====================================================
    # Inference
    # =====================================================
    def predict_windows(
        self,
        df_test: pd.DataFrame,
        return_reconstruction: bool = True
    ) -> Dict[str, Any]:
        """
        Create predict windows (1 hour blocks), reconstruct numeric outputs,
        and compute reconstruction error.
        """
        if self.model is None:
            raise RuntimeError("Model is not loaded/built.")

        X_pred, y_true, transformed_rows, pred_meta = self.preprocessor.transform_predict(
            df_test,
            return_metadata=True
        )

        if X_pred.shape[0] == 0:
            return {
                "X_pred": X_pred,
                "y_true": y_true,
                "y_pred": np.empty_like(y_true),
                "pred_meta": pred_meta,
                "window_errors": pd.DataFrame(),
                "signal_errors": pd.DataFrame(),
                "transformed_rows": transformed_rows,
            }

        y_pred = self.model.predict(X_pred, verbose=0)

        window_errors_df, signal_errors_df = self._compute_reconstruction_errors(
            y_true=y_true,
            y_pred=y_pred,
            pred_meta=pred_meta
        )

        result = {
            "X_pred": X_pred,
            "y_true": y_true,
            "y_pred": y_pred,
            "pred_meta": pred_meta,
            "window_errors": window_errors_df,
            "signal_errors": signal_errors_df,
            "transformed_rows": transformed_rows,
        }

        if not return_reconstruction:
            result.pop("X_pred", None)
            result.pop("y_true", None)
            result.pop("y_pred", None)

        return result

    def _compute_reconstruction_errors(
        self,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        pred_meta: pd.DataFrame
    ) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Compute:
        - per-window overall errors
        - per-window per-signal errors
        """
        signal_names = self.preprocessor.target_feature_names_

        abs_err = np.abs(y_true - y_pred)             # (n_win, seq, n_sig)
        sq_err = np.square(y_true - y_pred)

        # overall window metrics
        window_mae = abs_err.mean(axis=(1, 2))
        window_mse = sq_err.mean(axis=(1, 2))
        window_rmse = np.sqrt(window_mse)

        window_errors_df = pred_meta.copy()
        window_errors_df["window_mae"] = window_mae
        window_errors_df["window_mse"] = window_mse
        window_errors_df["window_rmse"] = window_rmse

        # signal-level metrics
        signal_mae = abs_err.mean(axis=1)   # (n_win, n_sig)
        signal_mse = sq_err.mean(axis=1)
        signal_rmse = np.sqrt(signal_mse)

        signal_error_rows = []
        for i in range(len(pred_meta)):
            base = pred_meta.iloc[i].to_dict()
            for j, sig in enumerate(signal_names):
                signal_error_rows.append({
                    **base,
                    "signal": sig,
                    "signal_mae": float(signal_mae[i, j]),
                    "signal_mse": float(signal_mse[i, j]),
                    "signal_rmse": float(signal_rmse[i, j]),
                })

        signal_errors_df = pd.DataFrame(signal_error_rows)

        return window_errors_df, signal_errors_df

    # =====================================================
    # Save / load
    # =====================================================
    def save(self, artifact_dir: str) -> None:
        """
        Save:
        - keras model
        - preprocessor object
        - model config
        - schema metadata
        """
        if self.model is None:
            raise RuntimeError("No model available to save.")

        os.makedirs(artifact_dir, exist_ok=True)

        # Save keras model
        model_path = os.path.join(artifact_dir, "model.keras")
        self.model.save(model_path)

        # Save preprocessor
        preprocessor_path = os.path.join(artifact_dir, "preprocessor.joblib")
        joblib.dump(self.preprocessor, preprocessor_path)

        # Save config
        config_path = os.path.join(artifact_dir, "model_config.json")
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(asdict(self.model_config), f, indent=2)

        # Save metadata
        metadata = {
            "input_feature_names": self.preprocessor.input_feature_names_,
            "target_feature_names": self.preprocessor.target_feature_names_,
            "train_window_size": self.preprocessor.config.train_window_size,
            "predict_window_size": self.preprocessor.config.predict_window_size,
        }
        metadata_path = os.path.join(artifact_dir, "metadata.json")
        with open(metadata_path, "w", encoding="utf-8") as f:
            json.dump(metadata, f, indent=2)

    @classmethod
    def load(cls, artifact_dir: str) -> "LSTMAutoencoderService":
        """
        Load saved service artifacts.
        """
        model_path = os.path.join(artifact_dir, "model.keras")
        preprocessor_path = os.path.join(artifact_dir, "preprocessor.joblib")
        config_path = os.path.join(artifact_dir, "model_config.json")

        model = tf.keras.models.load_model(model_path)
        preprocessor = joblib.load(preprocessor_path)

        with open(config_path, "r", encoding="utf-8") as f:
            cfg_dict = json.load(f)

        service = cls(
            preprocessor=preprocessor,
            model_config=LSTMAEModelConfig(**cfg_dict),
            model=model
        )
        return service

In [14]:
component = 'MOTOR'

model_config = LSTMAEModelConfig(
    latent_dim=2,
    encoder_lstm_1=2,
    encoder_lstm_2=2,
    decoder_lstm_1=2,
    decoder_lstm_2=2,
    dropout_rate=0.2,
    learning_rate=1e-3,
    batch_size=32,
    epochs=2,
    validation_split=0.2,
    early_stopping_patience=5,
)

service = LSTMAutoencoderService(
    preprocessor=preprocessor,
    model_config=model_config
)

train_result = service.fit(df_train)
service.save(f"models/autoencoders/cda/{component}")

Epoch 1/2
10274/10274 ━━━━━━━━━━━━━━━━━━━━ 334s 32ms/step - loss: 2.0559 - val_loss: 1.8394
Epoch 2/2
10274/10274 ━━━━━━━━━━━━━━━━━━━━ 331s 32ms/step - loss: 1.9069 - val_loss: 1.8050


In [15]:
service = LSTMAutoencoderService.load(f"models/autoencoders/cda/{component}")
pred_result = service.predict_windows(df_test)

window_errors = pred_result["window_errors"]
signal_errors = pred_result["signal_errors"]

Shape of predict windows: (14222, 60, 14), (14222, 60, 6)


In [22]:
import os
import json
import numpy as np
import pandas as pd

from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Tuple


@dataclass
class HealthIndexConfig:
    alpha: float = 1.0
    time_agg: str = "mean"      # mean, median, max, p95
    signal_agg: str = "rms"     # mean, rms, max
    unit_agg: str = "mean"      # mean, median, min, p10
    percentile_low: int = 50
    percentile_high: int = 95
    eps: float = 1e-8


class HealthIndexService:
    """
    Service to:
    - compute reference reconstruction error percentiles
    - persist them
    - score predicted windows
    - consolidate health index tables
    """

    def __init__(
        self,
        signal_cols: List[str],
        config: Optional[HealthIndexConfig] = None,
        error_stats: Optional[Dict[str, Dict[str, float]]] = None,
    ):
        self.signal_cols = signal_cols
        self.config = config or HealthIndexConfig()
        self.error_stats = error_stats

    # =====================================================
    # Fit reference stats
    # =====================================================
    def fit_error_stats(
        self,
        y_true: np.ndarray,
        y_pred: np.ndarray,
    ) -> Dict[str, Dict[str, float]]:
        """
        Compute per-signal reference percentiles from training reconstruction errors.
        """
        if y_true.shape != y_pred.shape:
            raise ValueError("y_true and y_pred must have the same shape")

        if y_true.shape[-1] != len(self.signal_cols):
            raise ValueError("signal_cols length does not match tensor last dimension")

        abs_err = np.abs(y_true - y_pred)

        p_low = self.config.percentile_low
        p_high = self.config.percentile_high

        stats = {}
        for j, sig in enumerate(self.signal_cols):
            sig_err = abs_err[:, :, j].reshape(-1)

            low = float(np.percentile(sig_err, p_low))
            high = float(np.percentile(sig_err, p_high))

            stats[sig] = {
                f"p{p_low}": low,
                f"p{p_high}": high,
                "mean": float(np.mean(sig_err)),
                "std": float(np.std(sig_err)),
                "min": float(np.min(sig_err)),
                "max": float(np.max(sig_err)),
                "n": int(sig_err.shape[0]),
            }

        self.error_stats = stats
        return stats

    # =====================================================
    # Persistence
    # =====================================================
    def save(self, artifact_dir: str) -> None:
        if self.error_stats is None:
            raise RuntimeError("error_stats are not fitted yet.")

        os.makedirs(artifact_dir, exist_ok=True)

        with open(os.path.join(artifact_dir, "health_index_config.json"), "w", encoding="utf-8") as f:
            json.dump(asdict(self.config), f, indent=2)

        with open(os.path.join(artifact_dir, "error_stats.json"), "w", encoding="utf-8") as f:
            json.dump(self.error_stats, f, indent=2)

        with open(os.path.join(artifact_dir, "signal_cols.json"), "w", encoding="utf-8") as f:
            json.dump(self.signal_cols, f, indent=2)

    @classmethod
    def load(cls, artifact_dir: str) -> "HealthIndexService":
        with open(os.path.join(artifact_dir, "health_index_config.json"), "r", encoding="utf-8") as f:
            config = HealthIndexConfig(**json.load(f))

        with open(os.path.join(artifact_dir, "error_stats.json"), "r", encoding="utf-8") as f:
            error_stats = json.load(f)

        with open(os.path.join(artifact_dir, "signal_cols.json"), "r", encoding="utf-8") as f:
            signal_cols = json.load(f)

        return cls(
            signal_cols=signal_cols,
            config=config,
            error_stats=error_stats,
        )

    # =====================================================
    # Scoring
    # =====================================================
    def score_windows(
        self,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        pred_meta: pd.DataFrame,
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """
        Score windows and return:
        - signal_window_df: one row per window per signal
        - window_hi_df: one row per window
        - unit_hi_df: one row per unit
        """
        if self.error_stats is None:
            raise RuntimeError("error_stats not available. Fit or load them first.")

        if y_true.shape != y_pred.shape:
            raise ValueError("y_true and y_pred must have the same shape")

        if y_true.shape[0] != len(pred_meta):
            raise ValueError("pred_meta must have one row per window")

        if y_true.shape[-1] != len(self.signal_cols):
            raise ValueError("signal_cols length does not match tensor last dimension")

        abs_err = np.abs(y_true - y_pred)
        n_windows, seq_len, n_signals = abs_err.shape

        def agg_time(arr, method: str):
            if method == "mean":
                return float(np.mean(arr))
            elif method == "median":
                return float(np.median(arr))
            elif method == "max":
                return float(np.max(arr))
            elif method == "p95":
                return float(np.percentile(arr, 95))
            else:
                raise ValueError("Invalid time_agg")

        def agg_signals(arr, method: str):
            if method == "mean":
                return float(np.mean(arr))
            elif method == "rms":
                return float(np.sqrt(np.mean(np.square(arr))))
            elif method == "max":
                return float(np.max(arr))
            else:
                raise ValueError("Invalid signal_agg")

        def agg_unit(series: pd.Series, method: str):
            if method == "mean":
                return series.mean()
            elif method == "median":
                return series.median()
            elif method == "min":
                return series.min()
            elif method == "p10":
                return series.quantile(0.10)
            else:
                raise ValueError("Invalid unit_agg")

        p_low = self.config.percentile_low
        p_high = self.config.percentile_high
        eps = self.config.eps

        signal_rows = []
        window_rows = []

        for i in range(n_windows):
            base_meta = pred_meta.iloc[i].to_dict()

            norm_vals_window = []
            raw_vals_window = []

            for j, sig in enumerate(self.signal_cols):
                sig_stats = self.error_stats[sig]
                low = sig_stats[f"p{p_low}"]
                high = sig_stats[f"p{p_high}"]

                if high <= low:
                    raise ValueError(f"Invalid error stats for signal '{sig}'")

                err_ts = abs_err[i, :, j]
                norm_ts = np.clip((err_ts - low) / (high - low + eps), a_min=0.0, a_max=None)

                raw_err = agg_time(err_ts, self.config.time_agg)
                norm_err = agg_time(norm_ts, self.config.time_agg)

                raw_vals_window.append(raw_err)
                norm_vals_window.append(norm_err)

                signal_rows.append({
                    **base_meta,
                    "signal": sig,
                    "recon_error_raw": raw_err,
                    "recon_error_norm": norm_err,
                    f"p{p_low}_ref": low,
                    f"p{p_high}_ref": high,
                })

            reconstruction_error_raw = agg_signals(np.array(raw_vals_window), self.config.signal_agg)
            reconstruction_error_score = agg_signals(np.array(norm_vals_window), self.config.signal_agg)
            health_index_window = float(np.exp(-self.config.alpha * reconstruction_error_score))

            window_rows.append({
                **base_meta,
                "reconstruction_error_raw": reconstruction_error_raw,
                "reconstruction_error_score": reconstruction_error_score,
                "health_index": health_index_window,
                "n_signals": n_signals,
                "window_size": seq_len,
            })

        signal_window_df = pd.DataFrame(signal_rows)
        window_hi_df = pd.DataFrame(window_rows)

        unit_col = "Unit" if "Unit" in window_hi_df.columns else "unit"

        unit_hi_df = (
            window_hi_df.groupby(unit_col, as_index=False)
            .agg(
                health_index=("health_index", lambda s: agg_unit(s, self.config.unit_agg)),
                reconstruction_error=("reconstruction_error_score", lambda s: agg_unit(s, self.config.unit_agg)),
                n_windows=("health_index", "count"),
                start_time=("start_time", "min"),
                end_time=("end_time", "max"),
            )
        )

        return signal_window_df, window_hi_df, unit_hi_df

    # =====================================================
    # Convenience output
    # =====================================================
    def consolidate_window_health_index(
        self,
        window_hi_df: pd.DataFrame,
        unit_col: str = "Unit",
        start_col: str = "start_time",
        end_col: str = "end_time",
        hi_col: str = "health_index",
    ) -> pd.DataFrame:
        """
        Return the compact dataframe you care about most:
        unit U between T0 and Tf had HI H
        """
        cols = [unit_col, start_col, end_col, hi_col]
        optional_cols = [c for c in ["hour_idx", "reconstruction_error_score", "created_fraction", "imputed_fraction"] if c in window_hi_df.columns]
        return window_hi_df[cols + optional_cols].copy()

In [24]:
train_pred = service.predict_windows(df_test, return_reconstruction=True)

hi_service = HealthIndexService(
    signal_cols=service.preprocessor.target_feature_names_,
    config=HealthIndexConfig(
        alpha=1.0,
        time_agg="mean",
        signal_agg="rms",
        unit_agg="mean",
        percentile_low=50,
        percentile_high=95,
    )
)

hi_service.fit_error_stats(
    y_true=train_pred["y_true"],
    y_pred=train_pred["y_pred"],
)

hi_service.save("models/autoencoders/cda/component_x/health_index")

Shape of predict windows: (14222, 60, 14), (14222, 60, 6)


In [25]:
hi_service = HealthIndexService.load("models/autoencoders/cda/component_x/health_index")

pred_result = service.predict_windows(df_test, return_reconstruction=True)

signal_window_df, window_hi_df, unit_hi_df = hi_service.score_windows(
    y_true=pred_result["y_true"],
    y_pred=pred_result["y_pred"],
    pred_meta=pred_result["pred_meta"],
)

compact_hi_df = hi_service.consolidate_window_health_index(window_hi_df)

Shape of predict windows: (14222, 60, 14), (14222, 60, 6)


Small architectural recommendation

I would store the HI artifacts inside the same model artifact folder, but in a separate subfolder, for example:

``` {python}
models/
  autoencoders/
    cda/
      component_x/
        model.keras
        preprocessor.joblib
        model_config.json
        metadata.json
        health_index/
          health_index_config.json
          error_stats.json
          signal_cols.json
```

This keeps things version-aligned while still separating responsibilities.

